# CRNN training on Colab GPU - handwritten Urdu

Full training run for the CRNN baseline on NUST-UHWR.

**Before starting:** `Runtime > Change runtime type > Hardware accelerator: GPU`.

Colab sessions disconnect without warning, so checkpoints are written to Google
Drive after every epoch and the run can be resumed from the last one.

Run the cells in order. Steps 1-5 are setup and take a few minutes; step 6 is the
training run itself.

## 1. Which GPU did this session get?

Record this output in `reports/experiment_log.md`. Colab assigns different
hardware each session, and training time is one of the metrics the project reports,
so a time without the GPU name beside it means nothing.

In [ ]:
import torch

print("torch        :", torch.__version__)
print("cuda available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise SystemExit("No GPU. Runtime > Change runtime type > GPU, then rerun.")

props = torch.cuda.get_device_properties(0)
print("device       :", props.name)
print("total VRAM   : %.1f GB" % (props.total_memory / 1024**3))

## 2. Mount Google Drive

Checkpoints and logs are written here so a disconnect does not lose the run.

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

DRIVE_DIR = Path("/content/drive/MyDrive/urdu_ocr")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print("Drive folder:", DRIVE_DIR)

## 3. Get the project code

**Option A - from GitHub.** Push the project first, then put the repository URL
below. This is the reproducible route: the notebook then records which commit
produced the result.

**Option B - upload a zip.** If the code is not on GitHub, zip the project folder
locally (excluding `.venv`, `data/raw` and `outputs`), upload it through the Files
panel on the left, and use the second cell instead.

In [ ]:
# Option A: from GitHub
REPO_URL = ""  # e.g. https://github.com/<user>/urdu-ocr-research.git

if REPO_URL:
    !git clone -q $REPO_URL /content/project
    %cd /content/project
    !git log -1 --format="commit %H%nfrom  %ci"
else:
    print("REPO_URL is empty. Use Option B below, or fill it in and rerun.")

In [ ]:
# Option B: from an uploaded zip. Skip this cell if Option A worked.
# Upload urdu_ocr_project.zip via the Files panel first.

# !mkdir -p /content/project
# !unzip -q /content/urdu_ocr_project.zip -d /content/project
# %cd /content/project

## 4. Install the dependencies Colab lacks

Colab already ships PyTorch, numpy, pandas, Pillow and PyYAML. **Do not reinstall
torch** - Colab's build matches its CUDA drivers, and replacing it usually breaks
the GPU. Only `jiwer` is missing.

The versions actually used are printed, and belong in the experiment log: they will
not match the pinned local versions, and that difference is part of the record.

In [ ]:
!pip install -q "jiwer==3.1.0"

import importlib.metadata as md

import torch

print("torch :", torch.__version__)
for package in ("numpy", "pandas", "pillow", "jiwer", "pyyaml"):
    try:
        print(f"{package:7}:", md.version(package))
    except md.PackageNotFoundError:
        print(f"{package:7}: MISSING")

## 5. Get the dataset and build the manifests

The archive is cached on Drive after the first download, so later sessions skip the
download. The MD5 is checked every time: a truncated download would otherwise
surface much later as mysterious training failures.

In [ ]:
import hashlib
import shutil
from pathlib import Path

EXPECTED_MD5 = "e367ff83c147671f4ae569bc63480af9"
ZENODO_URL = "https://zenodo.org/records/21674269/files/UHWR.zip?download=1"

cached = DRIVE_DIR / "UHWR.zip"
local = Path("/content/project/data/raw/nust_uhwr/UHWR.zip")
local.parent.mkdir(parents=True, exist_ok=True)

if cached.exists():
    print("Using the copy cached on Drive")
    shutil.copy(cached, local)
else:
    print("Downloading from Zenodo (159 MB)")
    !wget -q --show-progress -O "$local" "$ZENODO_URL"
    shutil.copy(local, cached)
    print("Cached to Drive for next time")

digest = hashlib.md5()
with local.open("rb") as handle:
    for chunk in iter(lambda: handle.read(1 << 20), b""):
        digest.update(chunk)

actual = digest.hexdigest()
print("md5:", actual)
assert actual == EXPECTED_MD5, f"MD5 mismatch: expected {EXPECTED_MD5}"
print("MD5 verified.")

In [ ]:
%cd /content/project
!unzip -q -o data/raw/nust_uhwr/UHWR.zip -d data/raw/nust_uhwr/

# Rebuild rather than trusting committed manifests: the paths inside them are
# relative, but rebuilding also re-applies the audit exclusions here.
!python -m src.data.prepare_dataset --dataset nust_uhwr \
    --root data/raw/nust_uhwr/UHWR --out data/manifests
!python -m src.data.vocabulary --dataset nust_uhwr --manifests data/manifests

## 5b. Verify the evaluation code in this environment

The metrics are what the thesis reports, and this is a different machine with
different package versions. Ten seconds well spent.

In [ ]:
!python scripts/test_evaluation.py | tail -4

## 5c. Short smoke run before the real one

Two minutes that confirm the GPU path, mixed precision and checkpointing all work.
Far cheaper than discovering a problem forty minutes into a full run.

In [ ]:
!python -m src.crnn.train --config configs/crnn_handwritten.yaml \
    --device cuda --amp --num-workers 2 \
    --limit-train 300 --limit-val 100 --epochs 2 --run-name smoke_colab

## 6. Full training run

All 8,483 training lines, validated on 1,060 each epoch.

The test split is not opened. Model selection uses validation CER alone.

If the session drops, rerun cells 1-5 and then the resume cell below rather than
this one.

In [ ]:
!python -m src.crnn.train --config configs/crnn_handwritten.yaml \
    --device cuda --amp --num-workers 2 \
    --epochs 30 --batch-size 16 \
    --run-name crnn_handwritten

In [ ]:
# Copy checkpoints, logs and predictions to Drive. Run this after training, and
# again any time the session looks unstable.
!mkdir -p "$DRIVE_DIR/outputs"
!cp -r outputs/* "$DRIVE_DIR/outputs/"
!ls -lh "$DRIVE_DIR/outputs/checkpoints/"

In [ ]:
# RESUME after a disconnect. Run cells 1-5 first, then this.
# !cp -r "$DRIVE_DIR/outputs" /content/project/
# !python -m src.crnn.train --config configs/crnn_handwritten.yaml \
#     --device cuda --amp --num-workers 2 --epochs 30 --batch-size 16 \
#     --run-name crnn_handwritten \
#     --resume outputs/checkpoints/crnn_handwritten_last.pt

## 7. Final evaluation on the held-out test split

**Run this once, after training is finished and no further tuning will happen.**
Seeing a test number and then changing a setting turns the test set into a second
validation set, and the reported result stops meaning what it claims.

In [ ]:
!python -m src.crnn.evaluate \
    --checkpoint outputs/checkpoints/crnn_handwritten_best.pt \
    --split test --device cuda

In [ ]:
# Save the final results to Drive, then download them for the write-up.
!cp -r outputs/* "$DRIVE_DIR/outputs/"
!cat outputs/metrics/crnn_handwritten_test.json

## 8. Record the run

Copy into `reports/experiment_log.md`:

| Field | Value |
| --- | --- |
| Date | |
| GPU and VRAM | from step 1 |
| torch version | from step 4 |
| Git commit | from step 3 |
| Epochs run, best epoch | from step 6 |
| Best validation CER | from step 6 |
| **Test CER and WER** | from step 7 |
| Training time, peak GPU memory | from step 6 |

Then download `outputs/` from Drive so the local repository holds the results.